# LoRA / QLoRA 概念デモノートブック

`lora_qlora.md` の理論を **numpy だけ** で手を動かして確認する。GPU不要・このまま実行可能。

扱うこと：
1. 低ランク分解 ΔW = B·A と再構成誤差
2. LoRA のパラメータ数削減（rを変えて観察）
3. Full / LoRA / QLoRA のメモリ内訳比較
4. NF4 量子化シミュレーション（int4 との誤差比較）
5. Double Quantization のオーバーヘッド計算

> 実機学習は `lora_qlora_finetune.ipynb`（Colab/GPU）で。

In [ ]:
# 依存（未インストールなら実行）
# %pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print('ready')

---

## 1. 低ランク分解 ΔW = B·A

微調整の重み更新 ΔW が低ランクなら、`B(d×r)·A(r×k)` で近似できる。
SVD でランクを切り詰めて再構成誤差を見る。

In [ ]:
d, k = 256, 256
# 内在ランクが低い ΔW を人工的に作る（真のランク=8）
true_rank = 8
B_true = np.random.randn(d, true_rank)
A_true = np.random.randn(true_rank, k)
dW = B_true @ A_true + 0.01*np.random.randn(d, k)  # わずかにノイズ

def low_rank_approx(M, r):
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    return (U[:, :r] * S[:r]) @ Vt[:r, :]

def rel_error(M, Mr):
    return np.linalg.norm(M - Mr) / np.linalg.norm(M)

ranks = [1, 2, 4, 8, 16, 32, 64, 128]
errs = [rel_error(dW, low_rank_approx(dW, r)) for r in ranks]
for r, e in zip(ranks, errs):
    print(f'r={r:3d}  相対誤差={e:.4f}  パラメータ数 r*(d+k)={r*(d+k):,}')

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(ranks, errs, 'o-')
plt.axvline(true_rank, color='r', ls='--', label=f'真のランク={true_rank}')
plt.xlabel('LoRA rank r'); plt.ylabel('再構成の相対誤差')
plt.title('低ランク近似: r が真のランクに達すると誤差が急減')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

---

## 2. LoRA のパラメータ数削減

`d=k=4096`（7B級）の Linear 1層で、Full と LoRA の学習パラメータ数を比較する。

In [ ]:
def full_params(d, k):  return d*k
def lora_params(d, k, r): return r*(d+k)

d = k = 4096
print(f'Full:           {full_params(d,k):,}')
for r in [4, 8, 16, 32, 64]:
    lp = lora_params(d, k, r)
    pct = 100*lp/full_params(d,k)
    print(f'LoRA (r={r:2d}):   {lp:>10,}   ({pct:.3f}% of full)')

In [ ]:
# 初期化の妙: B=0 なら学習開始時 ΔW=0 → 元モデルと同一出力
r = 8
A = np.random.randn(r, k) * (1/np.sqrt(k))   # kaiming風
Bz = np.zeros((d, r))                          # B=0 初期化
dW0 = Bz @ A
print('B=0 初期化での ||ΔW|| =', np.linalg.norm(dW0), '→ 学習開始時は元モデルと完全一致')

---

## 3. メモリ内訳: Full vs LoRA vs QLoRA

7B モデルで、重み・勾配・Adam state の内訳を計算して棒グラフにする（活性化は除く）。

In [ ]:
N = 7e9              # 7B params
trainable_ratio = 0.004   # LoRA: 約0.4%

def breakdown(mode):
    if mode == 'Full FT (bf16)':
        w = 2*N                     # bf16 重み
        g = 2*N                     # 勾配（全パラメータ）
        opt = 8*N                   # Adam m,v を fp32 (4+4)
    elif mode == 'LoRA (bf16)':
        w = 2*N                     # 重みは凍結だが保持
        g = 2*N*trainable_ratio
        opt = 8*N*trainable_ratio
    elif mode == 'QLoRA (NF4)':
        w = 0.5*N + 0.127/8*N       # 4bit重み + DoubleQuant定数
        g = 2*N*trainable_ratio
        opt = 8*N*trainable_ratio
    return np.array([w, g, opt]) / 1e9   # GB

modes = ['Full FT (bf16)', 'LoRA (bf16)', 'QLoRA (NF4)']
data = {m: breakdown(m) for m in modes}
for m in modes:
    w,g,o = data[m]
    print(f'{m:16s} 重み={w:6.2f}GB 勾配={g:6.3f}GB Adam={o:6.3f}GB 合計={w+g+o:6.2f}GB')

In [ ]:
labels = ['重み', '勾配', 'Adam state']
x = np.arange(len(modes)); bottom = np.zeros(len(modes))
plt.figure(figsize=(8,5))
for i, lab in enumerate(labels):
    vals = np.array([data[m][i] for m in modes])
    plt.bar(x, vals, bottom=bottom, label=lab)
    bottom += vals
plt.xticks(x, modes); plt.ylabel('GPUメモリ (GB)')
plt.title('7B 微調整のメモリ内訳（活性化除く）')
plt.legend(); plt.grid(True, axis='y', alpha=0.3)
for i, m in enumerate(modes):
    plt.text(i, bottom[i]+1, f'{bottom[i]:.1f}GB', ha='center')
plt.show()

---

## 4. NF4 量子化シミュレーション

NN の重みは平均0の正規分布に近い。NF4 は**正規分布の分位点**に合わせた非等間隔の4bit格子。
同じ4bitの int4（等間隔）と再構成誤差を比較する。

In [ ]:
# QLoRA の NF4 16格子点（bitsandbytes create_normal_map 由来）
NF4 = np.array([
    -1.0, -0.6961928, -0.5250731, -0.3949175, -0.2844414, -0.1847734,
    -0.0910500,  0.0,  0.0795803,  0.1609302,  0.2461123,  0.3379152,
     0.4407098,  0.5626170,  0.7229568,  1.0])
INT4 = np.linspace(-1, 1, 16)   # 等間隔（同じ16値=4bit）

def quantize_blockwise(x, levels, block=64):
    x = x.flatten().astype(np.float64)
    out = np.empty_like(x)
    for s in range(0, len(x), block):
        blk = x[s:s+block]
        absmax = np.abs(blk).max() + 1e-12
        norm = blk / absmax                       # [-1,1] に正規化
        idx = np.abs(norm[:,None] - levels[None,:]).argmin(1)  # 最近傍格子
        out[s:s+block] = levels[idx] * absmax     # 逆量子化
    return out

W = np.random.randn(4096*64)   # 正規分布の重み
W_nf4  = quantize_blockwise(W, NF4)
W_int4 = quantize_blockwise(W, INT4)
mse = lambda a,b: np.mean((a-b)**2)
print(f'NF4  再構成MSE = {mse(W, W_nf4):.6e}')
print(f'int4 再構成MSE = {mse(W, W_int4):.6e}')
print(f'NF4 は int4 より誤差 {mse(W,W_int4)/mse(W,W_nf4):.2f}x 小さい')

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(W, bins=120, density=True, alpha=0.4, label='重み分布 N(0,1)')
for v in NF4:  plt.axvline(v, color='g', alpha=0.5, lw=0.8)
for v in INT4: plt.axvline(v, color='r', alpha=0.5, lw=0.8, ls='--')
plt.plot([],[], color='g', label='NF4格子(分位点)')
plt.plot([],[], color='r', ls='--', label='int4格子(等間隔)')
plt.xlim(-3,3); plt.legend(); plt.title('NF4は密度の高い中央に格子点を集中させる')
plt.xlabel('正規化後の重み値'); plt.show()

---

## 5. Double Quantization のオーバーヘッド

block-wise だとブロックごとに量子化定数(scale)が要る。その bit/param を計算する。

In [ ]:
def scale_overhead_bits(block, scale_bytes):
    return scale_bytes*8 / block   # 1ブロックあたり scale を block個で割る

block = 64
naive = scale_overhead_bits(block, 4)            # fp32 scale
# Double Quant: scaleを8bitに量子化 + さらに上位scale(256ブロックごとfp32)
dq = scale_overhead_bits(block, 1) + scale_overhead_bits(block*256, 4)
print(f'素朴(fp32 scale):        {naive:.3f} bit/param')
print(f'Double Quant:            {dq:.3f} bit/param')
print(f'節約:                    {naive-dq:.3f} bit/param')
print(f'7Bなら {(naive-dq)*7e9/8/1e9:.2f} GB の追加削減')

---

## 6. まとめ: 数字で振り返る

このノートで確認したこと：
- ΔW は真のランク到達でほぼ誤差0 → **低ランク近似が妥当**
- r=8 で学習パラメータは **0.39%** に
- QLoRA は 7B を **約3.8GB** まで圧縮（Full 84GB → 1/20以下）
- NF4 は int4 より再構成誤差が小さい（重みが正規分布だから）

次は `lora_qlora_finetune.ipynb` で実際に微調整を回す。